# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [ ]:
import re, math, time, unicodedata, statistics, random, json
from array import array
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 220)


PALABRA = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower().replace("\u2019", "'"))
    return texto.encode("ascii", "ignore").decode("ascii")

def tokenizar(texto):
    return PALABRA.findall(normalizar(texto))

def recortar(texto):
    ini = re.search(r"\*{3}\s*START OF (?:THE|THIS) PROJECT GUTENBERG[^\n]*\n", texto, re.I)
    if ini:
        texto = texto[ini.end():]
    fin = re.search(r"\*{3}\s*END OF (?:THE|THIS) PROJECT GUTENBERG", texto, re.I)
    if fin:
        texto = texto[:fin.start()]
    return texto

# Localización del corpus 
RUTA_CORPUS = None   

def buscar_carpeta_epub():
    if RUTA_CORPUS:
        return Path(RUTA_CORPUS)
    casa = Path.home()
    for base in (casa / "Downloads", casa / "Descargas", casa / "Desktop", Path.cwd()):
        for sub in (("txt-files", "txt-files", "cache", "epub"),
                    ("txt-files", "cache", "epub"),
                    ("cache", "epub")):
            ruta = base.joinpath(*sub)
            if ruta.is_dir():
                return ruta
    raise FileNotFoundError("No se encontró la carpeta cache/epub, define RUTA_CORPUS")

def elegir_txt(carpeta):
    candidatos = [f for f in carpeta.glob("*.txt") if f.stat().st_size > 0]
    if not candidatos:
        return None
    principal = carpeta / f"pg{carpeta.name}.txt"
    if principal in candidatos:
        return principal
    return min(candidatos, key=lambda f: len(f.name))

CARPETA_EPUB = buscar_carpeta_epub()
carpetas = sorted((c for c in CARPETA_EPUB.iterdir() if c.is_dir() and c.name.isdigit()),
                  key=lambda c: int(c.name))
libros = {}
for carpeta in carpetas:
    archivo = elegir_txt(carpeta)
    if archivo is not None:
        libros[int(carpeta.name)] = archivo
print("Documentos válidos en Gutenberg:", len(libros))

N_DOCS = 1000     
SEMILLA = 2026

def leer_texto(id_libro):
    """Texto original completo (se vuelve a leer del disco cuando hace falta un fragmento)."""
    ruta = libros[id_libro]
    codificacion = "latin-1" if ruta.name.lower().endswith("-8.txt") else "utf-8"
    return ruta.read_bytes().decode(codificacion, errors="ignore")

def extraer_titulo(texto, id_libro):
    m = re.search(r"^\ufeff?Title:[ \t]*(.+(?:\n[ \t]+\S.*)*)", texto[:6000], re.M)
    if m:
        return re.sub(r"\s+", " ", m.group(1)).strip()
    return f"pg{id_libro}"

muestra = random.Random(SEMILLA).sample(sorted(libros), min(N_DOCS, len(libros)))

unicas = {}
corpus, titulos = {}, {}     
t0 = time.perf_counter()
for k, id_libro in enumerate(muestra, 1):
    texto = leer_texto(id_libro)
    titulos[id_libro] = extraer_titulo(texto, id_libro)
    corpus[id_libro] = [unicas.setdefault(p, p) for p in tokenizar(recortar(texto))]
    if k % 100 == 0:
        print(f"{k} de {len(muestra)} libros procesados")
print(f"Carga y tokenización: {time.perf_counter() - t0:.1f} s")


ids = sorted(corpus)         
tabla_docs = pd.DataFrame({
    "Identificador": ids,
    "Título": [titulos[i] for i in ids],
    "Términos únicos": [len(set(corpus[i])) for i in ids],
})
display(tabla_docs.head(15))

longitudes = np.array([len(corpus[i]) for i in ids])
print("Número de documentos:", len(ids))
print("Tamaño del vocabulario:", f"{len(unicas):,}")
print("Longitud (tokens) mínima / media / máxima:",
      f"{longitudes.min():,} / {longitudes.mean():,.0f} / {longitudes.max():,}")

Documentos válidos en Gutenberg: 78145
100 de 1000 libros procesados
200 de 1000 libros procesados
300 de 1000 libros procesados
400 de 1000 libros procesados
500 de 1000 libros procesados
600 de 1000 libros procesados
700 de 1000 libros procesados
800 de 1000 libros procesados
900 de 1000 libros procesados
1000 de 1000 libros procesados
Carga y tokenización: 38.7 s


,Identificador,Título,Términos únicos
0,29,The 1990 United States Census [incomplete],1248
1,290,The Stark Munro Letters,7949
2,361,Miss Billy — Married,6200
3,680,The Golden Threshold,2220
4,709,The Princess and Curdie,4909
5,733,History of the Decline and Fall of the Roman Empire — Volume 3,17775
6,795,"Ballads and Lyrics of Old France, with Other Poems",2799
7,997,Divina Commedia di Dante: Inferno,6769
8,1041,Shakespeare's Sonnets,3353
9,1313,Over the Sliprails,5355


Número de documentos: 1000
Tamaño del vocabulario: 953,264
Longitud (tokens) mínima / media / máxima: 4 / 60,740 / 2,268,893


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [4]:
def calcular_tf(tokens):
    """TF(t,d) = f(t,d): número de apariciones de cada término en el documento."""
    return Counter(tokens)

def calcular_df(corpus):
    """DF(t): en cuántos documentos del corpus aparece t (cada documento cuenta una sola vez)."""
    df = Counter()
    for tokens in corpus.values():
        df.update(set(tokens))
    return df

def calcular_idf(df, N):
    """IDF(t) = ln(N / DF(t)) + 1  (sklearn con smooth_idf=False)."""
    return {t: math.log(N / d) + 1 for t, d in df.items()}

def calcular_tfidf(tf, idf):
    """w(t,d) = TF(t,d) * IDF(t) para los términos del documento."""
    return {t: f * idf[t] for t, f in tf.items()}

# Prueba a mano con un corpus mínimo de 3 documentos
mini = {"d1": ["gato", "juega", "gato"],
        "d2": ["perro", "juega"],
        "d3": ["gato", "duerme"]}
df_mini = calcular_df(mini)
idf_mini = calcular_idf(df_mini, len(mini))
tf_mini = {d: calcular_tf(toks) for d, toks in mini.items()}
w_mini = {d: calcular_tfidf(tf_mini[d], idf_mini) for d in mini}

filas = []
for t in sorted(df_mini):
    fila = {"término": t, "DF": df_mini[t], "IDF": round(idf_mini[t], 4)}
    for d in mini:
        fila[f"TF {d}"] = tf_mini[d].get(t, 0)
        fila[f"w {d}"] = round(w_mini[d].get(t, 0.0), 4)
    filas.append(fila)
display(pd.DataFrame(filas))
# Comprobación manual: "gato" en d1 -> TF=2, DF=2, IDF=ln(3/2)+1=1.4055, w=2.8109

,término,DF,IDF,TF d1,w d1,TF d2,w d2,TF d3,w d3
0,duerme,1,2.0986,0,0.0000,0,0.0000,1,2.0986
1,gato,2,1.4055,2,2.8109,0,0.0000,1,1.4055
2,juega,2,1.4055,1,1.4055,1,1.4055,0,0.0000
3,perro,1,2.0986,0,0.0000,1,2.0986,0,0.0000


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [ ]:
# 1. Vocabulario con orden definido alfabético
# 3. DF cada documento cuenta una vez por término
df_corpus = calcular_df(corpus)
N = len(ids)
vocabulario = sorted(df_corpus)
pos = {t: j for j, t in enumerate(vocabulario)}
V = len(vocabulario)

# 4. IDF con la función de la sección anterior
idf = calcular_idf(df_corpus, N)
idf_vec = np.array([idf[t] for t in vocabulario])

# 2. Frecuencias TF -> matriz dispersa documento-término filas = ids, columnas = vocabulario
indptr, indices, datos = array("q", [0]), array("i"), array("d")
for doc in ids:
    tf = calcular_tf(corpus[doc])
    fila = sorted((pos[t], f) for t, f in tf.items())
    indices.extend(j for j, _ in fila)
    datos.extend(f for _, f in fila)
    indptr.append(len(indices))

TF = sparse.csr_matrix((np.frombuffer(datos, dtype=np.float64),
                        np.frombuffer(indices, dtype=np.int32),
                        np.frombuffer(indptr, dtype=np.int64)), shape=(N, V))

# 5. Matriz TF-IDF: w(t,d) = TF(t,d) * IDF(t)
W = (TF @ sparse.diags(idf_vec)).tocsr()

print("Matriz TF-IDF:", W.shape, "| valores no nulos:", f"{W.nnz:,}",
      f"| densidad: {W.nnz / (N * V):.4%}")

# Comprobación con las funciones de la sección 2 primer documento, 5 términos con mayor peso
d0 = ids[0]
w_d0 = calcular_tfidf(calcular_tf(corpus[d0]), idf)
top = sorted(w_d0.items(), key=lambda x: -x[1])[:5]
display(pd.DataFrame([{"término": t, "TF": int(TF[0, pos[t]]), "DF": df_corpus[t],
                       "IDF": idf[t], "TF-IDF (dict)": w, "TF-IDF (matriz)": W[0, pos[t]]}
                      for t, w in top]))

Matriz TF-IDF: (1000, 953264) | valores no nulos: 6,537,458 | densidad: 0.6858%


,término,TF,DF,IDF,TF-IDF (dict),TF-IDF (matriz)
0,percent,750,30,4.506558,3379.918423,3379.918423
1,units,751,66,3.718101,2792.293503,2792.293503
2,hispanic,200,3,6.809143,1361.828598,1361.828598
3,households,350,70,3.659260,1280.741013,1280.741013
4,housing,301,53,3.937463,1185.176473,1185.176473


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [6]:
TOL = 1e-9

# El vectorizador recibe exactamente los mismos tokens: analyzer devuelve la lista tal cual,
# así no aplica ningún preprocesamiento propio.
vectorizador = TfidfVectorizer(
    analyzer=lambda tokens: tokens,
    use_idf=True, smooth_idf=False, sublinear_tf=False, norm=None,
)
W_sk = vectorizador.fit_transform([corpus[i] for i in ids])

print("Misma forma:", W.shape == W_sk.shape, W.shape, W_sk.shape)
print("Mismo vocabulario y orden de columnas:",
      list(vectorizador.get_feature_names_out()) == vocabulario)

dif_max = abs(W - W_sk).max()
print("Máxima diferencia absoluta entre ambas matrices:", dif_max)
print("Dentro de la tolerancia (1e-9):", dif_max < TOL)
assert dif_max < TOL

print("Diferencia máxima entre IDF propios y vectorizador:",
      np.abs(idf_vec - vectorizador.idf_).max())

Misma forma: True (1000, 953264) (1000, 953264)
Mismo vocabulario y orden de columnas: True
Máxima diferencia absoluta entre ambas matrices: 0.0
Dentro de la tolerancia (1e-9): True
Diferencia máxima entre IDF propios y vectorizador: 0.0


**Interpretación.** Si la diferencia es del orden de 1e-12 o menor, la implementación explícita y `TfidfVectorizer` producen la misma matriz: tienen el mismo vocabulario y orden de columnas, la misma fórmula de IDF (`ln(N/DF)+1`, `smooth_idf=False`) y ninguna normalización. Las diferencias residuales son solo errores de redondeo en punto flotante.

### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [ ]:
consultas = {
    "Q1": {"necesidad": "Relatos de viajes por mar: barcos, capitanes y tormentas.",
           "texto": "ocean voyage ship captain storm"},
    "Q2": {"necesidad": "Novelas o relatos policiales sobre un asesinato y su investigación.",
           "texto": "murder detective crime investigation"},
    "Q3": {"necesidad": "Libros de cocina con recetas de pan, mantequilla y horno.",
           "texto": "recipes cooking bread butter oven"},
    "Q4": {"necesidad": "Historia del Imperio romano: emperadores, legiones y guerras.",
           "texto": "roman empire emperor legions war"},
}

# Normas de los documentos 
normas_docs = np.sqrt(W.multiply(W).sum(axis=1)).A1

def vector_consulta(texto):
    """Vector TF-IDF de la consulta con el vocabulario y los IDF del corpus (sin recalcular IDF)."""
    tf_q = Counter(t for t in tokenizar(texto) if t in pos)   # términos fuera del vocabulario se ignoran
    cols = sorted(pos[t] for t in tf_q)
    vals = [tf_q[vocabulario[j]] * idf_vec[j] for j in cols]
    return sparse.csr_matrix((vals, ([0] * len(cols), cols)), shape=(1, V))

def similitud_coseno(q, W, normas_docs):
    """sim(q,d) = (q·d) / (||q|| ||d||) para todos los documentos."""
    punto = (W @ q.T).toarray().ravel()
    norma_q = np.sqrt(q.multiply(q).sum())
    den = norma_q * normas_docs
    return np.divide(punto, den, out=np.zeros_like(punto), where=den > 0)

vectores_q, sims_exp, sims_sk = {}, {}, {}
for qid, c in consultas.items():
    tokens_q = tokenizar(c["texto"])
    fuera = [t for t in tokens_q if t not in pos]
    q = vector_consulta(c["texto"])
    q_sk = vectorizador.transform([tokens_q])                       # transform del vectorizador ajustado
    vectores_q[qid] = q
    sims_exp[qid] = similitud_coseno(q, W, normas_docs)             # implementación explícita
    sims_sk[qid] = cosine_similarity(q_sk, W_sk).ravel()            # scikit-learn
    print(f"{qid}: {c['necesidad']}")
    print(f"    consulta: {c['texto']!r} | tokens: {tokens_q} | fuera del vocabulario: {fuera}")
    print(f"    dif. máx. vector consulta: {abs(q - q_sk).max():.2e} | "
          f"dif. máx. similitudes: {np.abs(sims_exp[qid] - sims_sk[qid]).max():.2e}")
    assert np.allclose(sims_exp[qid], sims_sk[qid], atol=TOL)


def detalle_par(qid, doc_id):
    q = vectores_q[qid]
    d = W[ids.index(doc_id)]
    wq = dict(zip(q.indices, q.data))
    wd = dict(zip(d.indices, d.data))
    comunes = sorted(set(wq) & set(wd))
    tabla = pd.DataFrame({
        "Término compartido": [vocabulario[j] for j in comunes],
        "q_i": [wq[j] for j in comunes],
        "d_i": [wd[j] for j in comunes],
        "Producto parcial q_i·d_i": [wq[j] * wd[j] for j in comunes],
    })
    punto = tabla["Producto parcial q_i·d_i"].sum()
    nq = math.sqrt(sum(v * v for v in q.data))
    nd = math.sqrt(sum(v * v for v in d.data))
    return tabla, punto, nq, nd

qid_demo = "Q1"
doc_demo = ids[int(np.argmax(sims_exp[qid_demo]))]
tabla_par, punto, nq, nd = detalle_par(qid_demo, doc_demo)
display(tabla_par)
print(f"Consulta {qid_demo} vs documento {doc_demo} - {titulos[doc_demo]}")
print(f"Producto punto = {punto:.6f}")
print(f"||q|| = {nq:.6f} | ||d|| = {nd:.6f}")
print(f"Similitud = {punto:.6f} / ({nq:.6f} x {nd:.6f}) = {punto / (nq * nd):.6f}")
print(f"cosine_similarity de sklearn = {sims_sk[qid_demo][ids.index(doc_demo)]:.6f}")

Q1: Relatos de viajes por mar: barcos, capitanes y tormentas.
    consulta: 'ocean voyage ship captain storm' | tokens: ['ocean', 'voyage', 'ship', 'captain', 'storm'] | fuera del vocabulario: []
    dif. máx. vector consulta: 0.00e+00 | dif. máx. similitudes: 8.23e-16
Q2: Novelas o relatos policiales sobre un asesinato y su investigación.
    consulta: 'murder detective crime investigation' | tokens: ['murder', 'detective', 'crime', 'investigation'] | fuera del vocabulario: []
    dif. máx. vector consulta: 0.00e+00 | dif. máx. similitudes: 2.06e-16
Q3: Libros de cocina con recetas de pan, mantequilla y horno.
    consulta: 'recipes cooking bread butter oven' | tokens: ['recipes', 'cooking', 'bread', 'butter', 'oven'] | fuera del vocabulario: []
    dif. máx. vector consulta: 0.00e+00 | dif. máx. similitudes: 2.71e-16
Q4: Historia del Imperio romano: emperadores, legiones y guerras.
    consulta: 'roman empire emperor legions war' | tokens: ['roman', 'empire', 'emperor', 'legions', 'w

,Término compartido,q_i,d_i,Producto parcial q_i·d_i
0,captain,1.809681,79.625964,144.097594
1,ship,1.783072,21.396863,38.152144


Consulta Q1 vs documento 73145 - The thought-feeders
Producto punto = 182.249738
||q|| = 4.122053 | ||d|| = 599.147875
Similitud = 182.249738 / (4.122053 x 599.147875) = 0.073794
cosine_similarity de sklearn = 0.073794


**Interpretación.** El vector de la consulta usa el vocabulario y los IDF del corpus,  los términos que no están en el vocabulario se ignoran. 
Como `norm=None`, el coseno se obtiene dividiendo el producto punto entre las dos normas, y por eso la similitud queda entre 0 y 1 y no depende de la longitud del documento.

### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [ ]:
def fragmento(doc_id, q, ancho=150):
    """Fragmento del texto original alrededor del término de la consulta con mayor aporte al producto punto."""
    d = W[ids.index(doc_id)]
    wd = dict(zip(d.indices, d.data))
    aportes = sorted(((wd[j] * v, vocabulario[j]) for j, v in zip(q.indices, q.data) if j in wd),
                     reverse=True)
    texto = recortar(leer_texto(doc_id))
    for _, termino in aportes:
        m = re.search(r"\b" + re.escape(termino) + r"\b", texto, re.I)
        if m:
            ini, fin = max(0, m.start() - ancho), min(len(texto), m.end() + ancho)
            return "..." + re.sub(r"\s+", " ", texto[ini:fin]).strip() + "..."
    return re.sub(r"\s+", " ", texto[:2 * ancho]).strip() + "..."

def ranking_top(qid, k=5):
    sims = sims_exp[qid]
    orden = np.argsort(-sims, kind="stable")[:k]
    return pd.DataFrame([{
        "Posición": r,
        "Identificador": ids[i],
        "Título": titulos[ids[i]],
        "Puntuación": round(float(sims[i]), 6),
        "Fragmento": fragmento(ids[i], vectores_q[qid]),
    } for r, i in enumerate(orden, 1)])

rankings = {}
for qid, c in consultas.items():
    rankings[qid] = ranking_top(qid)
    
    top_exp = np.argsort(-sims_exp[qid], kind="stable")[:5]
    top_sk = np.argsort(-sims_sk[qid], kind="stable")[:5]
    assert np.allclose(sims_exp[qid][top_exp], sims_sk[qid][top_sk], atol=TOL)
    print(f"\n{qid}: {c['texto']!r} - {c['necesidad']}")
    print("Mismo top-5 con ambas versiones:", list(top_exp) == list(top_sk))
    display(rankings[qid])


Q1: 'ocean voyage ship captain storm' - Relatos de viajes por mar: barcos, capitanes y tormentas.
Mismo top-5 con ambas versiones: True


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,73145,The thought-feeders,0.073794,"...risen from the ground eight miles in the air. ""It works!"" he beamed in ecstasy, watching the altimeter. ""It's a neat job,"" admitted the pilot, Captain Lewis Hawes, condescendingly. Inventors were nice, but they we..."
1,2,61497,The Eagles Gather,0.067033,"...boats along. They're coming home now--those that have power enough in their engines to make the trip. My name's Duane, Jim Duane."" ""They call me Captain,"" said the little man. ""I've got other names, but mostly I a..."
2,3,74528,"An essay on the preservation of shipwrecked mariners in answer to the prize-questions proposed by the Royal Humane Society : ""1. What are the best means of preserving mariners from shipwreck?-2. Of keeping the vessel...",0.043120,"...ore! _Second Case of Shipwreck near Shore._ According to an extract of a letter from Basseterre, of Aug. 3, 1792, the ship Britannia, captain Woodyear, with thirty persons, including two women and a child on board..."
3,4,33282,"The Boy Pilot of the Lakes; Or, Nat Morton's Perils",0.041551,...59 IX A NARROW ESCAPE 67 X SAM SHAW APPEARS 74 XI CAPTAIN MARSHALL IS ANGRY 81 XII THE INVESTIGATION 88 XIII MAKING A CHANGE...
4,5,23374,The Dingo Boys: The Squatters of Wallaby Range,0.038432,"...s come out of his shell, eh, Tim?"" ""Yes,"" said Artemus, solemnly. ""Australian chief magnificently attired in a small piece of dirty cotton."" Captain Bedford, retired officer of the Royal Engineers, a bluff, slight..."



Q2: 'murder detective crime investigation' - Novelas o relatos policiales sobre un asesinato y su investigación.
Mismo top-5 con ambas versiones: True


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,29331,The Crevice,0.033321,"...of carrying it through, successfully. I shall send at once for the Master Mind."" ""The Master Mind?"" ""Yes, dear--Henry Blaine, the most eminent detective the English-speaking world has produced."" ""I have heard of h..."
1,2,9974,The Yellow Streak,0.022079,...must. But I don’t want you to help me. I don’t want you to shield me. I shall make it my business to see that Bude’s evidence is brought before the detective inspector from Scotland Yard who is being brought down ...
2,3,14818,The Daughter of Anderson Crow,0.016593,"...ustice New York Dodd, Mead and Company 1907 [Illustration: Anderson Crow] CONTENTS CHAPTER I. ANDERSON CROW, DETECTIVE II. THE PURSUIT BEGINS III. THE CULPRITS IV. ANDERSON RECTIFIES AN ERROR V. THE BABE ON THE DO..."
3,4,76632,A vocabulary of criminal slang with some examples of common usages,0.015416,"...BY LOUIS E. JACKSON Assisted by C. R. HELLYER, _City Detective Department_ PORTLAND, OREGON Price, $1.50 DEDI..."
4,5,70463,"The Pennsylvania Journal of prison discipline and philanthropy (Vol. VIII, No. III, July 1853)",0.015372,"...s periodical gives a large amount of information on Prison Discipline, and cannot fail to interest such as grieve over the sufferings occasioned by crime, and regard the imprisoned criminal as still belonging to o..."



Q3: 'recipes cooking bread butter oven' - Libros de cocina con recetas de pan, mantequilla y horno.
Mismo top-5 con ambas versiones: True


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,42631,Friendship Club Cook Book,0.062493,"...sliced 2 cloves garlic 6 large tomatoes 6 sweet green peppers 2 or 3 sprigs thyme 2 sprigs parsley 1 bay leaf 2 T butter 2 T flour 2 C stock or water salt pepper Disjoint chicken, season well with salt and pepper...."
1,2,64687,"To Be ""Talked About"" Use Cookie 'n' Cracker Cookin'",0.059666,"...d serve in a dip bowl placed on a snack tray. Arrange TOWN HOUSE CRACKERS around the bowl. It’s a pleasing surprise to find the crackers served hot—butter, place on cookie sheet and bake for 7 minutes at 350° F. G..."
2,3,79407,"Uncooked foods & how to use them a treatise on how to get the highest form of animal energy from food, with recipes for preparation, healthful combinations and menus",0.035748,"...A TREATISE ON HOW TO GET THE HIGHEST FORM OF ANIMAL ENERGY FROM FOOD With Recipes for Preparation, Healthful Combinations and Menus :: :: :: BY _Mr..."
3,4,47694,The Labour-saving House,0.031945,"...9 viii. Coal, Coke, and Gas: How to Use Them to the Best Advantage 141 ix. The Electric House. Cooking, Heating, Cleaning and Lighting by Electricity 171 A Final Word 187..."
4,5,61719,Food and Flavor: A Gastronomic Guide to Health and Good Living,0.031748,"...e American--Why do we Eat Poultry?--Is cold storage a Blessing?--Spoiling the American Oyster--""Smoked"" ham, bacon and fish--Flavor in Butter--Sweet Butter versus Salt. IV THE SCIENCE OF SAVORY COOKING 117 Desirab..."



Q4: 'roman empire emperor legions war' - Historia del Imperio romano: emperadores, legiones y guerras.
Mismo top-5 con ambas versiones: True


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,3574,Recollections of the Private Life of Napoleon — Volume 07,0.058299,"...of the Corps Legislatif, and at noon on the 29th set out a second time for Bayonne. The Empress, who to her great chagrin could not accompany the Emperor, sent for me on the morning of his departure, and renewed i..."
1,2,9831,The Court of the Empress Josephine,0.046516,...RESS AT MAYENCE XXI. THE RETURN OF THE EMPRESS TO PARIS XXII. THE DEATH OF THE YOUNG NAPOLEON XXIII. THE END OF THE WAR XXIV. THE EMPEROR'S RETURN XXV. THE COURT AT FONTAINEBLEAU XXVI. THE END OF THE YEAR 1807 I. ...
2,3,34397,A Short History of Germany,0.045787,...magne--France and Germany Separated--Feudal System CHAPTER V. Early Conditions--Hungarian Invasions--Creation of Burgs--Knighthood--Pope and Emperor Become Rivals--Henry IV.--Canossa--First Hohenstaufen--Welf and ...
3,4,2156,China and the Manchus,0.038970,"...-chêns had become subject to the neighbouring Kitans, then under the rule of the vigorous Kitan chieftain, Opaochi, who, in 907, proclaimed himself Emperor of an independent kingdom with the dynastic title of Liao..."
4,5,75682,Waterloo,0.036391,"...y slope, will know how to appreciate the immense difficulties under which the French labored. WATERLOO CHAPTER I In 1815 the Emperor was no longer a lean, sinewy, tireless, eternally vigilant human tiger--the Napo..."
